# Vegetation-Aware Flood Modeling

Vegetation controls how water moves across a landscape. Dense forest slows overland flow and holds water deeper locally, while bare or paved surfaces let water move fast and shallow. The SCS curve number method captures the other side: vegetation and soil type together determine how much rainfall infiltrates vs. runs off.

This notebook compares the standard flood analysis tools (uniform roughness, manually assigned curve numbers) with the new vegetation-aware functions that derive hydraulic parameters directly from land cover or NDVI data:

| Standard approach | Vegetation-aware approach |
|---|---|
| `travel_time(fl, slope, mannings_n=0.03)` | `vegetation_roughness()` &rarr; `travel_time()` |
| `curve_number_runoff(rain, curve_number=80)` | `vegetation_curve_number()` &rarr; `curve_number_runoff()` |
| `flood_depth(hand, water_level=10)` | `flood_depth_vegetation(hand, slope, n, q)` |

We'll use the same Copernicus DEM tile as the [flood analysis guide](12_Flood_Analysis.ipynb), with synthetic NLCD land cover and soil group rasters to demonstrate the full pipeline.

[Setup](#Setup) · [Vegetation roughness](#Vegetation-roughness) · [Travel time comparison](#Travel-time-comparison) · [Curve number from land cover](#Curve-number-from-land-cover) · [Runoff comparison](#Runoff-comparison) · [Vegetation-adjusted flood depth](#Vegetation-adjusted-flood-depth) · [Full pipeline comparison](#Full-pipeline-comparison)

## Setup

Load a DEM, compute the hydrology stack (fill, flow direction, accumulation, HAND), and build synthetic land cover / soil rasters.

In [ ]:
%matplotlib inline
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap, LogNorm, BoundaryNorm
from matplotlib.patches import Patch

import xrspatial
from xrspatial.flood import (
    flood_depth, inundation, curve_number_runoff, travel_time,
    vegetation_roughness, vegetation_curve_number, flood_depth_vegetation,
    NLCD_MANNINGS_N, NLCD_CURVE_NUMBER,
)

In [ ]:
try:
    import rasterio
    from rasterio.windows import Window

    url = (
        "https://copernicus-dem-30m.s3.amazonaws.com/"
        "Copernicus_DSM_COG_10_N46_00_W123_00_DEM/"
        "Copernicus_DSM_COG_10_N46_00_W123_00_DEM.tif"
    )

    with rasterio.open(url) as src:
        window = Window(col_off=2400, row_off=2400, width=600, height=600)
        data = src.read(1, window=window).astype(np.float64)
        nodata = src.nodata

    if nodata is not None:
        data[data == nodata] = np.nan

    H, W = data.shape
    dem = xr.DataArray(data, dims=['y', 'x'], name='elevation',
                       attrs={'res': (1, 1)})
    dem['y'] = np.linspace(H - 1, 0, H)
    dem['x'] = np.linspace(0, W - 1, W)
    print(f"Loaded Copernicus 30m DEM: {dem.shape}")

except Exception as e:
    print(f"Remote DEM unavailable ({e}), generating synthetic terrain")
    H, W = 600, 600
    dem = xr.DataArray(np.zeros((H, W)), dims=['y', 'x'])
    dem = dem.xrs.generate_terrain(seed=10)
    dem.name = 'elevation'
    print(f"Generated terrain: {dem.shape}")

In [ ]:
# Fill, resolve flats, flow direction, accumulation, HAND
dem_filled = xrspatial.fill(dem)
rng = np.random.RandomState(42)
dem_filled.values += rng.uniform(0, 0.001, dem_filled.shape)
dem_filled = xrspatial.fill(dem_filled)
rng2 = np.random.RandomState(123)
dem_filled.values += rng2.uniform(0, 1e-6, dem_filled.shape)

flow_dir = xrspatial.flow_direction(dem_filled)
flow_accum = xrspatial.flow_accumulation(flow_dir)

threshold = 200
hand_raster = xrspatial.hand(flow_dir, flow_accum, dem_filled, threshold=threshold)
slope_raster = xrspatial.slope(dem_filled)
fl_raster = xrspatial.flow_length(flow_dir)
hillshade = xrspatial.hillshade(dem)

# Stream network for overlays
streams = xr.where(flow_accum >= threshold, flow_accum, np.nan)
streams.name = 'streams'

print(f"HAND range: {np.nanmin(hand_raster.values):.1f} to {np.nanmax(hand_raster.values):.1f} m")
print(f"Slope range: {np.nanmin(slope_raster.values):.2f} to {np.nanmax(slope_raster.values):.1f} deg")

### Synthetic land cover and soil rasters

We build a plausible NLCD land cover raster from the terrain itself: valley floors get developed classes, mid-slopes get forest, ridgetops get grassland/shrub. This is crude but gives spatially varying roughness and curve numbers that track the terrain in a realistic way.

The soil group raster assigns sandy soil (group A) on ridges and clay (group D) in valleys, reflecting typical catena patterns.

In [ ]:
hand_vals = hand_raster.fillna(0).values

# NLCD classes based on HAND (proximity to drainage)
nlcd_data = np.full((H, W), 71, dtype=np.int32)  # default: Grassland
nlcd_data[hand_vals < 5]  = 24   # Developed, High Intensity (floodplain)
nlcd_data[(hand_vals >= 5) & (hand_vals < 15)] = 21   # Developed, Open Space
nlcd_data[(hand_vals >= 15) & (hand_vals < 40)] = 41  # Deciduous Forest
nlcd_data[(hand_vals >= 40) & (hand_vals < 80)] = 42  # Evergreen Forest
nlcd_data[hand_vals >= 80] = 52   # Shrub/Scrub (ridgetops)

nlcd_raster = xr.DataArray(nlcd_data, dims=['y', 'x'], name='nlcd',
                           coords=dem.coords, attrs=dem.attrs)

# Soil group: valleys=D (clay), mid=C/B, ridges=A (sandy)
sg_data = np.full((H, W), 2, dtype=np.int32)  # default: B
sg_data[hand_vals < 10]  = 4   # D (clay in valley bottoms)
sg_data[(hand_vals >= 10) & (hand_vals < 30)] = 3  # C
sg_data[(hand_vals >= 30) & (hand_vals < 60)] = 2  # B
sg_data[hand_vals >= 60] = 1   # A (sandy ridgetops)

sg_raster = xr.DataArray(sg_data, dims=['y', 'x'], name='soil_group',
                         coords=dem.coords, attrs=dem.attrs)

# Show the land cover map
nlcd_labels = {24: 'Dev High', 21: 'Dev Open', 41: 'Deciduous',
               42: 'Evergreen', 52: 'Shrub', 71: 'Grassland'}
nlcd_colors = {24: '#eb1a1d', 21: '#e8d1aa', 41: '#6ca966',
               42: '#1d6533', 52: '#ccb879', 71: '#e8e87b'}

codes_in_order = [24, 21, 41, 42, 52, 71]
cmap_nlcd = ListedColormap([nlcd_colors[c] for c in codes_in_order])
bounds = [0] + [codes_in_order[i] + 0.5 for i in range(len(codes_in_order))]
# Map codes to sequential integers for plotting
nlcd_plot = np.full_like(nlcd_data, np.nan, dtype=np.float64)
for i, code in enumerate(codes_in_order):
    nlcd_plot[nlcd_data == code] = i
nlcd_plot_da = xr.DataArray(nlcd_plot, dims=['y', 'x'], coords=dem.coords)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Land cover
ax = axes[0]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
nlcd_plot_da.plot.imshow(ax=ax, cmap=cmap_nlcd, alpha=0.7, add_colorbar=False,
                        vmin=-0.5, vmax=len(codes_in_order) - 0.5)
ax.legend(handles=[Patch(facecolor=nlcd_colors[c], label=nlcd_labels[c])
                   for c in codes_in_order],
          loc='lower right', fontsize=9, framealpha=0.9)
ax.set_title('Synthetic NLCD land cover')
ax.set_axis_off()

# Soil group
ax = axes[1]
sg_cmap = ListedColormap(['#fee5d9', '#fcae91', '#fb6a4a', '#cb181d'])
sg_plot = xr.DataArray(sg_data.astype(np.float64), dims=['y', 'x'], coords=dem.coords)
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
sg_plot.plot.imshow(ax=ax, cmap=sg_cmap, alpha=0.7, add_colorbar=False,
                    vmin=0.5, vmax=4.5)
ax.legend(handles=[Patch(facecolor=c, label=f'Group {l}')
                   for c, l in zip(['#fee5d9', '#fcae91', '#fb6a4a', '#cb181d'],
                                   ['A (sandy)', 'B', 'C', 'D (clay)'])],
          loc='lower right', fontsize=9, framealpha=0.9)
ax.set_title('Hydrologic soil group')
ax.set_axis_off()

plt.tight_layout()

## Vegetation roughness

`vegetation_roughness` converts land cover or NDVI data into a Manning's n raster. Two modes are available:

- **`mode='nlcd'`**: Categorical lookup from NLCD codes. Each code maps to a literature-standard roughness value (Chow 1959, Arcement & Schneider 1989). Unrecognized codes produce NaN.
- **`mode='ndvi'`**: Piecewise linear interpolation. Bare ground (NDVI < 0.1) gets low n (~0.02), dense canopy (NDVI > 0.6) gets high n (~0.16). Less precise than categorical, but works with any multispectral imagery.

Compare this with the standard approach of picking a single uniform n value.

In [ ]:
# Derive spatially varying Manning's n from NLCD
n_raster = vegetation_roughness(nlcd_raster, mode='nlcd')

n_vals = n_raster.values[~np.isnan(n_raster.values)]
print(f"Manning's n from NLCD:")
print(f"  min:    {n_vals.min():.3f}")
print(f"  median: {np.median(n_vals):.3f}")
print(f"  max:    {n_vals.max():.3f}")
print()

# Print the lookup table
print("NLCD code -> Manning's n:")
for code in sorted(NLCD_MANNINGS_N):
    count = np.sum(nlcd_data == code)
    if count > 0:
        print(f"  {code:2d}: n={NLCD_MANNINGS_N[code]:.3f}  ({count:>6d} cells)")

In [ ]:
n_cmap = LinearSegmentedColormap.from_list('roughness',
    ['#ffffcc', '#a1dab4', '#41b6c4', '#225ea8'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Uniform n
ax = axes[0]
uniform_n = xr.DataArray(np.full((H, W), 0.03, dtype=np.float64),
                         dims=['y', 'x'], coords=dem.coords)
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
uniform_n.plot.imshow(ax=ax, cmap=n_cmap, alpha=0.7, add_colorbar=True,
                      vmin=0.02, vmax=0.16,
                      cbar_kwargs={'label': "Manning's n", 'shrink': 0.7})
ax.set_title('Standard: uniform n = 0.03')
ax.set_axis_off()

# Vegetation-derived n
ax = axes[1]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
n_raster.plot.imshow(ax=ax, cmap=n_cmap, alpha=0.7, add_colorbar=True,
                     vmin=0.02, vmax=0.16,
                     cbar_kwargs={'label': "Manning's n", 'shrink': 0.7})
ax.set_title('Vegetation-aware: n from NLCD')
ax.set_axis_off()

plt.tight_layout()

### NDVI mode

When you have multispectral imagery instead of a classified land cover map, use `mode='ndvi'`. We'll generate a synthetic NDVI raster that correlates with elevation (higher = sparser vegetation) to demonstrate.

In [ ]:
# Synthetic NDVI: dense vegetation in valleys, sparse on ridges
elev_norm = (dem_filled.values - np.nanmin(dem_filled.values)) / \
            (np.nanmax(dem_filled.values) - np.nanmin(dem_filled.values))
ndvi_data = np.clip(0.7 - 0.5 * elev_norm + np.random.RandomState(99).normal(0, 0.05, (H, W)),
                    -0.1, 0.9).astype(np.float64)
ndvi_raster = xr.DataArray(ndvi_data, dims=['y', 'x'], name='ndvi',
                           coords=dem.coords, attrs=dem.attrs)

n_from_ndvi = vegetation_roughness(ndvi_raster, mode='ndvi')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ndvi_cmap = LinearSegmentedColormap.from_list('ndvi', ['#8c510a', '#d8b365', '#f6e8c3',
                                                       '#c7eae5', '#5ab4ac', '#01665e'])
ndvi_raster.plot.imshow(ax=ax, cmap=ndvi_cmap, add_colorbar=True,
                        cbar_kwargs={'label': 'NDVI', 'shrink': 0.7})
ax.set_title('Synthetic NDVI')
ax.set_axis_off()

ax = axes[1]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
n_from_ndvi.plot.imshow(ax=ax, cmap=n_cmap, alpha=0.7, add_colorbar=True,
                        vmin=0.02, vmax=0.16,
                        cbar_kwargs={'label': "Manning's n", 'shrink': 0.7})
ax.set_title("Manning's n from NDVI")
ax.set_axis_off()

plt.tight_layout()

## Travel time comparison

The existing `travel_time` function accepts Manning's n as either a scalar or a DataArray. With `vegetation_roughness`, the n raster plugs in directly. The effect is pronounced: forested slopes slow water down, while developed areas near channels let it move faster.

We compare the time of concentration (maximum travel time across the grid) for three scenarios:
1. Uniform n = 0.03 (channel-like, standard assumption)
2. Uniform n = 0.10 (forested, conservative assumption)
3. Spatially varying n from NLCD (the vegetation-aware approach)

In [ ]:
tt_uniform_low  = travel_time(fl_raster, slope_raster, mannings_n=0.03)
tt_uniform_high = travel_time(fl_raster, slope_raster, mannings_n=0.10)
tt_vegetation   = travel_time(fl_raster, slope_raster, mannings_n=n_raster)

for label, tt in [('Uniform n=0.03', tt_uniform_low),
                  ('Uniform n=0.10', tt_uniform_high),
                  ('NLCD vegetation', tt_vegetation)]:
    vals = tt.values[np.isfinite(tt.values)]
    print(f"{label:>20s}:  Tc = {vals.max():.1f},  median = {np.median(vals):.1f}")

In [ ]:
tt_cmap = LinearSegmentedColormap.from_list('tt', ['#08306b', '#2171b5', '#fee08b', '#d73027'])

# Shared color scale from the vegetation result
tt_finite = tt_vegetation.values[np.isfinite(tt_vegetation.values)]
vmin, vmax = np.nanpercentile(tt_finite, [2, 98])

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, tt, title in zip(axes,
                         [tt_uniform_low, tt_uniform_high, tt_vegetation],
                         ['Uniform n=0.03', 'Uniform n=0.10', 'Vegetation n (NLCD)']):
    hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    tt.plot.imshow(ax=ax, cmap=tt_cmap, alpha=0.75, vmin=vmin, vmax=vmax,
                   add_colorbar=True,
                   cbar_kwargs={'label': 'Travel time', 'shrink': 0.7})
    ax.set_title(title)
    ax.set_axis_off()

plt.tight_layout()

The vegetation-aware result splits the difference: developed valley floors move water fast (like n=0.03), while forested mid-slopes slow it down (closer to n=0.10). This gives a more realistic spatial pattern than either uniform assumption.

## Curve number from land cover

`vegetation_curve_number` automates the NLCD + soil group &rarr; CN lookup that the [flood analysis guide](12_Flood_Analysis.ipynb) did manually. The lookup table contains 60 entries (15 NLCD classes &times; 4 soil groups) from TR-55 and HEC-HMS.

Compare:
- **Before**: manually build a CN raster with `xr.where` (error-prone, limited to a few classes)
- **After**: one function call that handles all 15 NLCD classes and 4 soil groups

In [ ]:
cn_raster = vegetation_curve_number(nlcd_raster, sg_raster)

cn_vals = cn_raster.values[~np.isnan(cn_raster.values)]
print(f"Curve number from NLCD + soil group:")
print(f"  min:    {cn_vals.min():.0f}")
print(f"  median: {np.median(cn_vals):.0f}")
print(f"  max:    {cn_vals.max():.0f}")
print()

# Compare with the manual approach from the flood analysis guide
hand_filled = hand_raster.fillna(0)
cn_manual = xr.where(hand_filled < 20, 85.0, 55.0).astype(np.float64)
cn_manual.name = 'curve_number_manual'

print(f"Manual CN (2-class):  {np.unique(cn_manual.values)}")
print(f"Vegetation CN range:  {cn_vals.min():.0f} to {cn_vals.max():.0f} ({len(np.unique(cn_vals))} distinct values)")

In [ ]:
cn_cmap = LinearSegmentedColormap.from_list('cn', ['#1a9850', '#fee08b', '#d73027'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
cn_manual_da = xr.DataArray(cn_manual.values, dims=['y', 'x'], coords=dem.coords)
cn_manual_da.plot.imshow(ax=ax, cmap=cn_cmap, alpha=0.7, vmin=25, vmax=100,
                         add_colorbar=True,
                         cbar_kwargs={'label': 'Curve Number', 'shrink': 0.7})
ax.set_title('Manual: 2-class CN (55 / 85)')
ax.set_axis_off()

ax = axes[1]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
cn_raster.plot.imshow(ax=ax, cmap=cn_cmap, alpha=0.7, vmin=25, vmax=100,
                      add_colorbar=True,
                      cbar_kwargs={'label': 'Curve Number', 'shrink': 0.7})
ax.set_title('Vegetation-aware: CN from NLCD + soil')
ax.set_axis_off()

plt.tight_layout()

## Runoff comparison

Feed both CN rasters into `curve_number_runoff` with a uniform 100 mm storm. The vegetation-aware CN produces a more nuanced runoff pattern: developed areas on clay soils (high CN) generate much more runoff than forested ridges on sandy soil (low CN).

In [ ]:
rainfall = xr.DataArray(
    np.full((H, W), 100.0, dtype=np.float64),
    dims=['y', 'x'], coords=dem.coords, attrs=dem.attrs, name='rainfall',
)

runoff_manual = curve_number_runoff(rainfall, curve_number=cn_manual_da)
runoff_veg    = curve_number_runoff(rainfall, curve_number=cn_raster)

for label, ro in [('Manual 2-class CN', runoff_manual),
                  ('Vegetation CN', runoff_veg)]:
    vals = ro.values[~np.isnan(ro.values)]
    print(f"{label:>20s}:  mean={vals.mean():.1f} mm,  "
          f"min={vals.min():.1f} mm,  max={vals.max():.1f} mm")

In [ ]:
runoff_cmap = LinearSegmentedColormap.from_list('runoff', ['#ffffcc', '#fd8d3c', '#800026'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
runoff_manual.plot.imshow(ax=ax, cmap=runoff_cmap, alpha=0.8, vmin=0, vmax=80,
                          add_colorbar=True,
                          cbar_kwargs={'label': 'Runoff (mm)', 'shrink': 0.7})
ax.set_title('Runoff: manual 2-class CN')
ax.set_axis_off()

ax = axes[1]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
runoff_veg.plot.imshow(ax=ax, cmap=runoff_cmap, alpha=0.8, vmin=0, vmax=80,
                       add_colorbar=True,
                       cbar_kwargs={'label': 'Runoff (mm)', 'shrink': 0.7})
ax.set_title('Runoff: vegetation-aware CN')
ax.set_axis_off()

plt.tight_layout()

## Vegetation-adjusted flood depth

The standard `flood_depth` uses a uniform water level: every channel gets the same stage. `flood_depth_vegetation` computes depth from Manning's normal depth equation instead, which factors in slope and roughness:

$$h = \left(\frac{q \cdot n}{\sqrt{\tan(S)}}\right)^{3/5}$$

where $q$ is unit discharge (m&sup2;/s), $n$ is Manning's roughness, and $S$ is slope in degrees. The flood depth is $h - \text{HAND}$ where $h > \text{HAND}$.

The key difference: rougher surfaces (denser vegetation) produce deeper, slower water locally. Steeper slopes produce shallower, faster water. This gives a physically grounded depth estimate that varies with both terrain and land cover.

In [ ]:
# Standard flood depth at 10 m water level
depth_standard = flood_depth(hand_raster, water_level=10)

# Vegetation-adjusted flood depth
# q=2.0 m^2/s is a moderate unit discharge
depth_veg = flood_depth_vegetation(hand_raster, slope_raster, n_raster,
                                   unit_discharge=2.0)

for label, d in [('Standard (wl=10m)', depth_standard),
                 ('Vegetation-aware', depth_veg)]:
    vals = d.values[~np.isnan(d.values)]
    n_flooded = len(vals)
    pct = 100 * n_flooded / (H * W)
    if len(vals) > 0:
        print(f"{label:>25s}:  {n_flooded:>6d} cells ({pct:.1f}%),  "
              f"median={np.median(vals):.2f} m,  max={vals.max():.2f} m")
    else:
        print(f"{label:>25s}:  no flooded cells")

In [ ]:
depth_cmap = LinearSegmentedColormap.from_list('depth', ['#ffffcc', '#fd8d3c', '#800026'])

# Get shared vmax from whichever has deeper water
d1 = depth_standard.values[~np.isnan(depth_standard.values)]
d2 = depth_veg.values[~np.isnan(depth_veg.values)]
depth_vmax = max(np.percentile(d1, 98) if len(d1) else 1,
                 np.percentile(d2, 98) if len(d2) else 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
depth_standard.plot.imshow(ax=ax, cmap=depth_cmap, alpha=0.85, vmin=0, vmax=depth_vmax,
                           add_colorbar=True,
                           cbar_kwargs={'label': 'Flood depth (m)', 'shrink': 0.7})
ax.set_title('Standard: uniform water level = 10 m')
ax.set_axis_off()

ax = axes[1]
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
depth_veg.plot.imshow(ax=ax, cmap=depth_cmap, alpha=0.85, vmin=0, vmax=depth_vmax,
                      add_colorbar=True,
                      cbar_kwargs={'label': 'Flood depth (m)', 'shrink': 0.7})
ax.set_title('Vegetation-aware: Manning normal depth')
ax.set_axis_off()

plt.tight_layout()

### Effect of roughness on flood depth

To isolate the roughness effect, we compare `flood_depth_vegetation` with low uniform n (bare ground) vs. the NLCD-derived n, keeping all other inputs the same.

In [ ]:
depth_bare = flood_depth_vegetation(hand_raster, slope_raster,
                                    mannings_n=0.025, unit_discharge=2.0)
depth_veg2 = flood_depth_vegetation(hand_raster, slope_raster,
                                    mannings_n=n_raster, unit_discharge=2.0)

for label, d in [('Bare (n=0.025)', depth_bare),
                 ('Vegetation (NLCD n)', depth_veg2)]:
    vals = d.values[~np.isnan(d.values)]
    if len(vals) > 0:
        print(f"{label:>25s}:  {len(vals):>6d} cells flooded,  "
              f"median={np.median(vals):.3f} m,  max={vals.max():.3f} m")

# Depth difference where both are flooded
both_flooded = ~np.isnan(depth_bare.values) & ~np.isnan(depth_veg2.values)
if both_flooded.any():
    diff = depth_veg2.values[both_flooded] - depth_bare.values[both_flooded]
    print(f"\nDepth difference (veg - bare) where both flooded:")
    print(f"  mean: {diff.mean():+.3f} m  (positive = vegetation makes it deeper)")
    print(f"  max:  {diff.max():+.3f} m")

## Full pipeline comparison

Putting it all together: a side-by-side of the standard workflow (uniform parameters) vs. the vegetation-aware workflow (parameters derived from land cover and soil data).

| Step | Standard | Vegetation-aware |
|------|----------|------------------|
| Roughness | n = 0.03 everywhere | `vegetation_roughness(nlcd)` |
| Curve number | CN = 80 everywhere | `vegetation_curve_number(nlcd, soil)` |
| Runoff | `curve_number_runoff(rain, 80)` | `curve_number_runoff(rain, cn_raster)` |
| Travel time | `travel_time(fl, slope, 0.03)` | `travel_time(fl, slope, n_raster)` |
| Flood depth | `flood_depth(hand, 10)` | `flood_depth_vegetation(hand, slope, n, q)` |

In [ ]:
# Standard pipeline
runoff_std = curve_number_runoff(rainfall, curve_number=80.0)
tt_std     = travel_time(fl_raster, slope_raster, mannings_n=0.03)
depth_std  = flood_depth(hand_raster, water_level=10)

# Vegetation-aware pipeline
n_veg      = vegetation_roughness(nlcd_raster, mode='nlcd')
cn_veg     = vegetation_curve_number(nlcd_raster, sg_raster)
runoff_va  = curve_number_runoff(rainfall, curve_number=cn_veg)
tt_va      = travel_time(fl_raster, slope_raster, mannings_n=n_veg)
depth_va   = flood_depth_vegetation(hand_raster, slope_raster, n_veg,
                                    unit_discharge=2.0)

print("Metric                      Standard    Vegetation-aware")
print("-" * 60)

ro_std_mean = np.nanmean(runoff_std.values)
ro_va_mean = np.nanmean(runoff_va.values)
print(f"Mean runoff (mm)            {ro_std_mean:>8.1f}    {ro_va_mean:>8.1f}")

tc_std = np.nanmax(tt_std.values[np.isfinite(tt_std.values)])
tc_va = np.nanmax(tt_va.values[np.isfinite(tt_va.values)])
print(f"Time of concentration       {tc_std:>8.1f}    {tc_va:>8.1f}")

d_std_vals = depth_std.values[~np.isnan(depth_std.values)]
d_va_vals = depth_va.values[~np.isnan(depth_va.values)]
print(f"Flooded cells               {len(d_std_vals):>8d}    {len(d_va_vals):>8d}")
if len(d_std_vals) > 0 and len(d_va_vals) > 0:
    print(f"Median flood depth (m)      {np.median(d_std_vals):>8.2f}    {np.median(d_va_vals):>8.2f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Row 1: Standard
titles_std = ['Standard: runoff', 'Standard: travel time', 'Standard: flood depth']
data_std = [runoff_std, tt_std, depth_std]
cmaps = [runoff_cmap, tt_cmap, depth_cmap]
labels = ['Runoff (mm)', 'Travel time', 'Flood depth (m)']

# Row 2: Vegetation-aware
titles_va = ['Vegetation: runoff', 'Vegetation: travel time', 'Vegetation: flood depth']
data_va = [runoff_va, tt_va, depth_va]

for row, (titles, datasets) in enumerate([(titles_std, data_std), (titles_va, data_va)]):
    for col, (title, dat, cmap, label) in enumerate(zip(titles, datasets, cmaps, labels)):
        ax = axes[row, col]
        hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
        dat.plot.imshow(ax=ax, cmap=cmap, alpha=0.8, add_colorbar=True,
                        cbar_kwargs={'label': label, 'shrink': 0.7})
        ax.set_title(title)
        ax.set_axis_off()

plt.tight_layout()

### Key takeaways

- **Spatially varying roughness** from `vegetation_roughness` gives a more realistic travel time pattern than any single uniform value. Forested slopes slow water; developed channels don't.
- **CN from land cover + soil** via `vegetation_curve_number` replaces manual `xr.where` logic with a single lookup call covering 15 NLCD classes and 4 soil groups. The resulting runoff pattern tracks land use in a way that a 2-class approximation can't.
- **`flood_depth_vegetation`** uses Manning's normal depth instead of a uniform water level, so flood depth responds to both roughness and slope. Higher roughness produces deeper, slower water. This won't replace a hydraulic model, but it's a step closer to physical realism.
- All three functions accept custom lookup tables, so you can substitute your own NLCD-to-n or NLCD-to-CN mapping without changing the pipeline.

### References

- Chow, V.T. (1959). Open Channel Hydraulics. Manning's n tables.
- Arcement, G.J. & Schneider, V.R. (1989). Guide for selecting Manning's roughness coefficients. USGS Water Supply Paper 2339.
- USDA-NRCS (1986). Urban Hydrology for Small Watersheds, TR-55. SCS Curve Number tables.
- Baptist, M.J. et al. (2007). On inducing equations for vegetation resistance. Journal of Hydraulic Research 45(4), 435-450.